# Multivariate Gaussian Visualizer

Chapter 13: Probability and Random Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Explore how mean and covariance shape a 2D Gaussian density.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-13/multivariate-gaussian/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
from torch.distributions import MultivariateNormal

torch.manual_seed(0)

# The demo starts at mean (0, 0), standard deviations 1 and 1, correlation 0.
# Here rho = 0.6 so the ellipses tilt; set it to 0 for the demo's circles.
mux, muy, sx, sy, rho = 0.0, 0.0, 1.0, 1.0, 0.6

mu = torch.tensor([mux, muy], dtype=torch.float64)
Sigma = torch.tensor([[sx ** 2, rho * sx * sy],
                      [rho * sx * sy, sy ** 2]], dtype=torch.float64)
dist = MultivariateNormal(mu, covariance_matrix=Sigma)

# ── Density by the formula vs torch ─────────────────────────────────────────
x = torch.tensor([0.5, -0.3], dtype=torch.float64)
d = x - mu
by_hand = math.exp(-0.5 * d @ torch.linalg.solve(Sigma, d)) / (2 * math.pi * math.sqrt(torch.linalg.det(Sigma)))
print("p(x) by formula =", by_hand, " torch =", dist.log_prob(x).exp().item())

# ── The covariance's eigenvectors are the ellipse axes ──────────────────────
evals, evecs = torch.linalg.eigh(Sigma)
print("\neigenvalues of Sigma =", evals.tolist(), "-> 1-sigma semi-axes", evals.sqrt().tolist())
print("major axis direction =", evecs[:, 1].tolist())

# ── Sampling: x = mu + L z with Sigma = L L^T (Cholesky), z ~ N(0, I) ───────
L = torch.linalg.cholesky(Sigma)
samples = mu + torch.randn(200000, 2, dtype=torch.float64) @ L.T
print("\nsample mean       =", samples.mean(0).tolist())
print("sample covariance =", torch.cov(samples.T).tolist())
print("sample correlation =", torch.corrcoef(samples.T)[0, 1].item(), "(target", rho, ")")

# Mahalanobis distance: about 39% of the mass lies inside the 1-sigma ellipse in 2D
m2 = ((samples - mu) @ torch.linalg.inv(Sigma) * (samples - mu)).sum(1)
print("fraction inside the 1-sigma ellipse =", (m2 <= 1).double().mean().item(), " theory 1 - e^-1/2 =", 1 - math.exp(-0.5))

p(x) by formula = 0.13252506056567911  torch = 0.13252506056567911

eigenvalues of Sigma = [0.4, 1.6] -> 1-sigma semi-axes [0.6324555320336759, 1.2649110640673518]
major axis direction = [0.7071067811865475, 0.7071067811865475]

sample mean       = [0.0004402372875301499, -0.001992088632465976]
sample covariance = [[0.9946060655172873, 0.5977166381739064], [0.5977166381739064, 0.9982771318848748]]
sample correlation = 0.5998521670192551 (target 0.6 )
fraction inside the 1-sigma ellipse = 0.3951  theory 1 - e^-1/2 = 0.3934693402873666
